# Jedna partycja per device i wymuszony kierunek DESC clustering po event_time
Wszystkie dane urządzenia trafiają do jednej partycji, co sprawia, że odczyt staje się coraz wolniejszy.
Dane posortowane są w kolejności od najnowszych do najstarszych.
Partition key = device_id
Clustering key = event_time DESC

In [106]:
import uuid, time
from cassandra.cluster import Cluster

In [107]:
# Otwieranie połączenia
cluster = Cluster(['cassandra_nosql_lab'], port=9042)
session = cluster.connect()
print("Połączono z Cassandra")

Połączono z Cassandra


In [108]:
# Tworzenie Keyspace
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS lab_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
""")
print("Utworzono keyspace")

Utworzono keyspace


In [109]:
# Użycie Keyspace
session.set_keyspace('lab_keyspace')

# Tworzenie tabeli
session.execute("""
    CREATE TABLE IF NOT EXISTS events_by_device_3 (
        device_id TEXT,
        day TEXT,
        event_time TIMESTAMP,
        temperature FLOAT,
        humidity FLOAT,
        PRIMARY KEY (device_id, event_time)
    ) WITH CLUSTERING ORDER BY (event_time DESC)
""")
print("Utworzono tabelę 'users'")

Utworzono tabelę 'users'


## Generate test data

In [110]:
!python ../_data_generator/main.py \
  --loader cassandra \
  --table events_by_device_3 \
  --records 1000 \
  --batch-size 100

Done. Loader=cassandra, Records=1000


In [111]:
start = time.perf_counter()
rows = session.execute("""
SELECT *
FROM
    events_by_device_3
WHERE
    device_id='device_1'
LIMIT 10
""")
end = time.perf_counter()
print(f"Zapytanie wykonane w {end - start:.4f} sekund")
print("Znalezione rekordy:")
for row in rows:
    print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) - {row.event_time}")

Zapytanie wykonane w 0.0049 sekund
Znalezione rekordy:
- device_1 2026-03-28 (31.440000534057617°C, 39.9900016784668%) - 2026-03-28 16:12:12.043000
- device_1 2026-03-28 (34.880001068115234°C, 31.760000228881836%) - 2026-03-28 16:10:23.982000
- device_1 2026-03-28 (27.940000534057617°C, 61.25%) - 2026-03-28 16:10:19.627000
- device_1 2026-03-28 (20.3799991607666°C, 46.290000915527344%) - 2026-03-28 16:10:16.523000
- device_1 2026-03-28 (33.72999954223633°C, 43.099998474121094%) - 2026-03-28 16:09:19.629000
- device_1 2026-03-28 (32.13999938964844°C, 48.72999954223633%) - 2026-03-28 16:09:16.523000
- device_1 2026-03-28 (22.389999389648438°C, 39.209999084472656%) - 2026-03-28 16:08:19.630000
- device_1 2026-03-28 (24.420000076293945°C, 38.5099983215332%) - 2026-03-28 16:06:54.466000
- device_1 2026-03-28 (34.93000030517578°C, 56.689998626708984%) - 2026-03-28 16:06:16.522000
- device_1 2026-03-28 (28.170000076293945°C, 31.940000534057617%) - 2026-03-28 16:04:19.630000


In [112]:
# Zamknięcie połączenia
cluster.shutdown()